# **Section 1 - Dataset Overview**

In [ ]:

import pandas as pd

df = pd.read_excel("/content/E Commerce Dataset.xlsx")

df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

# **Section 2 - Data Cleaning**

**Step 1 — Fill Missing Numeric Values**

In [ ]:
numeric_cols = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder"
]

for col in numeric_cols:
    df[col].fillna(df[col].median(), inplace=True)

In [ ]:
df.isnull().sum()

**Step 3 — Check Duplicate Customers**

In [ ]:
df.duplicated().sum()

# **Section 3 - Feature Engineering**

**STEP 1 — Create ChurnFlag**

In [ ]:
df["ChurnFlag"] = df["Churn"]

**STEP 2 — Create TenureGroup**

In [ ]:
df["TenureGroup"] = pd.cut(
    df["Tenure"],
    bins=[-1,6,12,24,100],
    labels=["0-6 Months","7-12 Months","13-24 Months","24+ Months"]
)

**STEP 3 — Create EngagementScore**

In [ ]:
df["EngagementScore"] = (
    df["HourSpendOnApp"] +
    df["OrderCount"] +
    df["CouponUsed"]
) / 3

**STEP 4 — Create RevenueProxy**

In [ ]:
df["RevenueProxy"] = df["OrderCount"] * df["CashbackAmount"]

**STEP 5 — Create RiskSegment**

In [ ]:
df["RiskSegment"] = "Low"

df.loc[
    (df["SatisfactionScore"] <= 2) &
    (df["Complain"] == 1) &
    (df["Tenure"] < 6),
    "RiskSegment"
] = "High"

df.loc[
    (df["SatisfactionScore"] <= 3) &
    (df["Tenure"] < 12),
    "RiskSegment"
] = "Medium"

**STEP 6 — Verify New Columns**

In [ ]:
df.head()

In [ ]:
df["TenureGroup"].value_counts()

In [ ]:
df.shape

# **Section 4 - Statistical Validation**

**Step 1 — Prepare Data**

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest

**Step 2 — Extract Required Counts**

In [ ]:
# group data
grouped = df.groupby("Complain")["ChurnFlag"].agg(["count", "sum"])
grouped.columns = ["total", "churned"]

print(grouped)

**Step 3 — Perform Z-Test**

In [ ]:
# values
churned = grouped["churned"].values
total = grouped["total"].values

# z-test
stat, p_value = proportions_ztest(churned, total)

print("Z-stat:", stat)
print("P-value:", p_value)

The difference in churn rates between customers who raised complaints and those who did not is statistically significant, confirming complaints as a strong driver of churn.

# **Section 5 - Survival Analysis**

**Step 1 — Install Library**

In [ ]:
pip install lifelines

In [ ]:
from lifelines import KaplanMeierFitter
import matplotlib.pyplot as plt

kmf = KaplanMeierFitter()

# fit model
kmf.fit(durations=df["Tenure"], event_observed=df["ChurnFlag"])

# plot
kmf.plot()

plt.title("Customer Survival Curve")
plt.xlabel("Tenure (Months)")
plt.ylabel("Survival Probability")
plt.show()

Customer churn risk is heavily concentrated in the first 6–12 months, after which retention stabilises significantly.

# **Section 6 - A/B Testing Simulation**

**Step 1 — Create Groups**

In [ ]:
np.random.seed(42)

df["Group"] = np.random.choice(["Control", "Treatment"], size=len(df))

**Step 2 — Simulate Improvement**

In [ ]:
df["AdjustedChurn"] = df["ChurnFlag"]

df.loc[df["Group"] == "Treatment", "AdjustedChurn"] = (
    df["ChurnFlag"] * 0.8
)

**Step 3 — Compare Groups**

In [ ]:
result = df.groupby("Group")["AdjustedChurn"].mean()
print(result)

**Step 4 — Statistical Test**

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

control = df[df["Group"] == "Control"]
treatment = df[df["Group"] == "Treatment"]

count = [
    control["ChurnFlag"].sum(),
    treatment["AdjustedChurn"].sum()
]

nobs = [
    len(control),
    len(treatment)
]

stat, pval = proportions_ztest(count, nobs)

print("P-value:", pval)

A targeted retention strategy can reduce churn by ~3 percentage points, which translates to a meaningful revenue impact.

**STEP 4 — Export to SQL**

In [ ]:
df.to_csv("ecommerce_churn_clean.csv", index=False)